# Import thư viện, Khởi tạo và Metric

In [ ]:
import pandas as pd
import numpy as np
import os
import time
from sklearn.utils.class_weight import compute_class_weight

import gc
import rmm
import cudf
rmm.reinitialize(
    managed_memory=True,
)
import dask_cudf
import dask.array as da

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (
    classification_report, accuracy_score, balanced_accuracy_score,
    f1_score, precision_score, recall_score, matthews_corrcoef,
    cohen_kappa_score, confusion_matrix
)

from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import warnings
warnings.filterwarnings('ignore')
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [ ]:
import random
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

In [ ]:
class HybridDataset(Dataset):
    def __init__(self, X_seq, X_static, y):
        self.X_seq = torch.FloatTensor(X_seq)
        self.X_static = torch.FloatTensor(X_static)
        self.y = torch.LongTensor(y)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X_seq[idx], self.X_static[idx], self.y[idx]

class HybridBiLSTMModel(nn.Module):
    def __init__(self, input_dim_per_phase, static_dim):
        super().__init__()

        self.hidden_dim = 128
        self.num_layers = 1
        self.dropout_p = 0.3
        self.num_classes = 3

        self.bilstm = nn.LSTM(
            input_size=input_dim_per_phase,
            hidden_size=self.hidden_dim,
            num_layers=self.num_layers,
            batch_first=True,
            bidirectional=True
        )

        self.dropout = nn.Dropout(self.dropout_p)

        self.mlp = nn.Sequential(
            nn.Linear(static_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 32),
            nn.ReLU()
        )

        self.fc = nn.Linear(self.hidden_dim * 2 + 32, self.num_classes)

    def forward(self, x_seq, x_static):
        _, (h_n, _) = self.bilstm(x_seq)

        # h_n: (num_layers*2, batch, hidden_dim)
        h_forward = h_n[-2]
        h_backward = h_n[-1]

        h_concat = torch.cat((h_forward, h_backward), dim=1)
        h_concat = self.dropout(h_concat)

        static_out = self.mlp(x_static)

        combined = torch.cat((h_concat, static_out), dim=1)
        return self.fc(combined)

In [ ]:
# Metrics
def gmean_score(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred)
    per_class = []
    for i in range(cm.shape[0]):
        tp = cm[i,i]
        fn = cm[i].sum() - tp
        fp = cm[:,i].sum() - tp
        tn = cm.sum() - tp - fn - fp
        sens = tp / (tp + fn) if (tp + fn) > 0 else 0
        spec = tn / (tn + fp) if (tn + fp) > 0 else 0
        per_class.append(np.sqrt(sens * spec))
    return np.prod(per_class) ** (1/len(per_class)) if per_class else 0

def gmean_per_class(y_true, y_pred, target_class):
    cm = confusion_matrix(y_true, y_pred)
    i = target_class
    tp = cm[i,i]
    fn = cm[i].sum() - tp
    fp = cm[:,i].sum() - tp
    tn = cm.sum() - tp - fn - fp

    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    return np.sqrt(recall * specificity)

def print_results(
    version_name,
    phase,
    y_true,
    y_pred,
    time_build_model=None,
    time_predict=None
):
    target_names = ['Excellent', 'Good', 'Average']

    print(f"\n{'='*30} {version_name} - Phase {phase} {'='*30}")
    print(classification_report(
        y_true,
        y_pred,
        digits=10,
        target_names=target_names
    ))

    # Precision / Recall / F1 theo từng class
    prec_per_class = precision_score(y_true, y_pred, average=None)
    rec_per_class  = recall_score(y_true, y_pred, average=None)
    f1_per_class   = f1_score(y_true, y_pred, average=None)

    # G-Mean per class
    gmean_classes = [
        gmean_per_class(y_true, y_pred, i)
        for i in range(len(target_names))
    ]

    print("G-Mean per class (one-vs-rest):")
    for idx, name in enumerate(target_names):
        print(f"  {name:<10}: {gmean_classes[idx]:.10f}")

    print()

    # ===== TẠO DICTIONARY METRICS =====
    metrics = {
        'Version': version_name,
        'Phase': phase,

        'TimeBuildModel': time_build_model,
        'TimePredict': time_predict,

        'Accuracy': accuracy_score(y_true, y_pred),
        'BalancedAcc': balanced_accuracy_score(y_true, y_pred),

        'Precision Macro': precision_score(y_true, y_pred, average='macro'),
        'Precision Weighted': precision_score(y_true, y_pred, average='weighted'),

        'Recall Macro': recall_score(y_true, y_pred, average='macro'),
        'Recall Weighted': recall_score(y_true, y_pred, average='weighted'),

        'F1-Score Macro': f1_score(y_true, y_pred, average='macro'),
        'F1-Score Weighted': f1_score(y_true, y_pred, average='weighted'),

        'GMean': gmean_score(y_true, y_pred),

        'MCC': matthews_corrcoef(y_true, y_pred),
        'Kappa': cohen_kappa_score(y_true, y_pred),
    }

    # ===== THÊM METRIC CHO TỪNG CLASS =====
    for i, name in enumerate(target_names):
        metrics[f'Precision_{name}'] = prec_per_class[i]
        metrics[f'Recall_{name}'] = rec_per_class[i]
        metrics[f'F1-Score_{name}'] = f1_per_class[i]
        metrics[f'G-Mean_{name}'] = gmean_classes[i]

    # In ra console
    for k, v in metrics.items():
        if k not in ['Version', 'Phase'] and v is not None:
            print(f"{k:22} : {v:.10f}")

    return metrics

# Chuẩn bị dữ liệu + train

In [ ]:
def prepare_and_train_hybrid(train_path, val_path, device, version_name):

    print(f"Loading train (GPU): {train_path}")
    ddf_train = dask_cudf.read_parquet(train_path)

    df_val = pd.read_parquet(val_path, engine='pyarrow') if val_path else None

    train_len = len(ddf_train)
    print(f"Train samples: {train_len}")
    if df_val is not None:
        print(f"Validation samples: {len(df_val)}")

    cols_to_drop = ['user_id', 'course_id']
    ddf_train = ddf_train.drop(columns=[c for c in cols_to_drop if c in ddf_train.columns])

    if df_val is not None:
        df_val = df_val.drop(columns=cols_to_drop, errors='ignore')

    y_train = ddf_train['label_3'].compute().to_numpy()
    X_train_ddf = ddf_train.drop('label_3', axis=1)

    if df_val is not None:
        y_val = df_val['label_3'].values
        X_val_df = df_val.drop('label_3', axis=1)

    train_columns = X_train_ddf.columns.tolist()
    phase_cols = [c for c in train_columns if any(f"_p{p}_" in c for p in ['1','2','3','4'])]
    static_cols = [c for c in train_columns if c not in phase_cols]

    for p in ['1','2','3','4']:
        print(f"Phase {p}: {len([c for c in phase_cols if f'_p{p}_' in c])} features")

    def build_seq_dask(ddf, p_cols):
        phases = []
        for p in ['1','2','3','4']:
            cols = sorted([c for c in p_cols if f"_p{p}_" in c])
            phases.append(ddf[cols].compute().to_numpy())
        return np.stack(phases, axis=1)

    def build_seq_pandas(df, p_cols):
        phases = []
        for p in ['1','2','3','4']:
            cols = sorted([c for c in p_cols if f"_p{p}_" in c])
            phases.append(df[cols].values)
        return np.stack(phases, axis=1)

    X_seq_train = build_seq_dask(X_train_ddf, phase_cols)
    X_static_train = X_train_ddf[static_cols].compute().to_numpy()

    print(f"Time-series shape: {X_seq_train.shape}")
    print(f"Static feature shape: {X_static_train.shape}")

    scaler_seq = StandardScaler()
    N, T, F = X_seq_train.shape
    X_seq_train = scaler_seq.fit_transform(X_seq_train.reshape(-1, F)).reshape(N, T, F)

    scaler_static = StandardScaler()
    X_static_train = scaler_static.fit_transform(X_static_train)

    if df_val is not None:
        X_seq_val = build_seq_pandas(X_val_df, phase_cols)
        X_static_val = X_val_df[static_cols].values
        N2 = X_seq_val.shape[0]
        X_seq_val = scaler_seq.transform(X_seq_val.reshape(-1, F)).reshape(N2, T, F)
        X_static_val = scaler_static.transform(X_static_val)

    le = LabelEncoder()
    y_train_enc = le.fit_transform(y_train)
    print(f"Classes: {le.classes_}")

    if df_val is not None:
        y_val_enc = le.transform(y_val)

    train_loader = DataLoader(HybridDataset(X_seq_train, X_static_train, y_train_enc), batch_size=256, shuffle=True)

    if df_val is not None:
        val_loader = DataLoader(HybridDataset(X_seq_val, X_static_val, y_val_enc), batch_size=256, shuffle=False)

    model = HybridBiLSTMModel(F, X_static_train.shape[1]).to(device)

    unique_classes = np.unique(y_train_enc)
    weights = compute_class_weight('balanced', classes=unique_classes, y=y_train_enc)
    weights = torch.FloatTensor(weights).to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    best_loss = float('inf')
    patience = 10
    wait = 0
    start_train = time.perf_counter()

    for epoch in range(50):
        model.train()
        train_loss = 0
        for xb_seq, xb_static, yb in train_loader:
            xb_seq, xb_static, yb = xb_seq.to(device), xb_static.to(device), yb.to(device)
            optimizer.zero_grad()
            out = model(xb_seq, xb_static)
            loss = criterion(out, yb)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        train_loss /= len(train_loader)

        if df_val is not None:
            model.eval()
            val_loss = 0
            with torch.no_grad():
                for xb_seq, xb_static, yb in val_loader:
                    xb_seq, xb_static, yb = xb_seq.to(device), xb_static.to(device), yb.to(device)
                    val_loss += criterion(model(xb_seq, xb_static), yb).item()
            val_loss /= len(val_loader)
            print(f"Epoch {epoch+1}: train = {train_loss:.4f}, val = {val_loss:.4f}")
            monitor = val_loss
        else:
            print(f"Epoch {epoch+1}: loss = {train_loss:.4f}")
            monitor = train_loss

        if monitor < best_loss - 1e-4:
            best_loss = monitor
            best_state = model.state_dict()
            wait = 0
        else:
            wait += 1
            if wait >= patience: break

    model.load_state_dict(best_state)
    time_build = time.perf_counter() - start_train

    os.makedirs("saved_models", exist_ok=True)
    torch.save(model.state_dict(), f"saved_models/BiLSTM_{version_name}.pt")

    return model, scaler_seq, scaler_static, le, phase_cols, static_cols, time_build

# Chạy theo từng V

In [ ]:
base_path = "/kaggle/input/datasets/anhtran10/lo-dataset"

In [ ]:
def run_experiment(base_path, train_file, val_file, test_prefix, version_name, device=None):
    if device is None:
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    print(f"\n{'#'*20}")
    print(f"Version: {version_name}")
    print(f"{'#'*20}")

    model, scaler_seq, scaler_static, le, phase_cols, static_cols, time_build = \
        prepare_and_train_hybrid(f"{base_path_1}/{train_file}", f"{base_path}/{val_file}", device, version_name)

    results = []

    model.eval()

    for phase in range(1, 5):
        test_path = f"{base_path}/{test_prefix}_{phase}.parquet"
        print(f"\n--- Test Phase {phase}: {test_path} ---")

        df = pd.read_parquet(test_path, engine='pyarrow')
        df = df.drop(columns=['user_id','course_id'], errors='ignore')

        y_test_raw = df['label_3'].values
        X_df = df.drop('label_3', axis=1)

        def build_seq_local(df_input):
            phases_list = []
            for p in ['1','2','3','4']:
                cols = sorted([c for c in phase_cols if f"_p{p}_" in c])
                phases_list.append(df_input[cols].values)
            return np.stack(phases_list, axis=1)

        X_seq_test = build_seq_local(X_df)
        X_static_test = X_df[static_cols].values

        N, T, F = X_seq_test.shape
        X_seq_test = scaler_seq.transform(X_seq_test.reshape(-1, F)).reshape(N, T, F)
        X_static_test = scaler_static.transform(X_static_test)

        y_test_enc = le.transform(y_test_raw)

        test_dataset = HybridDataset(X_seq_test, X_static_test, y_test_enc)
        test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False)

        all_probs = []
        start_time = time.perf_counter()

        with torch.no_grad():
            for xb_seq, xb_static, _ in test_loader:
                xb_seq = xb_seq.to(device)
                xb_static = xb_static.to(device)

                outputs = model(xb_seq, xb_static)

                prob = torch.softmax(outputs, dim=1).cpu().numpy()
                all_probs.append(prob)

        time_pred = time.perf_counter() - start_time

        probs = np.vstack(all_probs)
        preds = np.argmax(probs, axis=1)

        metrics = print_results(version_name, phase, y_test_enc, preds, time_build, time_pred)

        os.makedirs("results_BiLSTM", exist_ok=True)

        cm = confusion_matrix(y_test_enc, preds)
        pd.DataFrame(cm).to_csv(f"results_BiLSTM/confusion_matrix_{version_name}_phase{phase}.csv", index=False)

        df_prob = pd.DataFrame(probs, columns=[f"Prob_Class_{i}" for i in range(probs.shape[1])])
        df_prob['y_true'] = y_test_enc
        df_prob['y_pred'] = preds
        df_prob.to_csv(f"results_BiLSTM/probability_matrix_{version_name}_phase{phase}.csv", index=False)

        results.append(metrics)

        del df, X_seq_test, X_static_test, all_probs, probs, preds
        torch.cuda.empty_cache()
        gc.collect()

    df_results = pd.DataFrame(results).round(10)

    ordered_cols = [
        "Version","Phase","TimeBuildModel","TimePredict","Accuracy","BalancedAcc",
        "Precision Macro","Precision Weighted","Recall Macro","Recall Weighted",
        "F1-Score Macro","F1-Score Weighted","GMean","MCC","Kappa",
        "Precision_Excellent","Recall_Excellent","F1-Score_Excellent","G-Mean_Excellent",
        "Precision_Good","Recall_Good","F1-Score_Good","G-Mean_Good",
        "Precision_Average","Recall_Average","F1-Score_Average","G-Mean_Average"
    ]

    df_results = df_results[[c for c in ordered_cols if c in df_results.columns]]

    return df_results

## V_Median

In [ ]:
base_path = "/kaggle/input/datasets/anhtran10/lo-dataset/Median/Median"

In [ ]:
df_v1 = run_experiment(
    base_path=base_path,
    train_file="train_median.parquet",
    val_file="val.parquet",
    test_prefix="test",
    version_name="V1 (Median)"
)
df_v1


####################
Version: V1 (Median)
####################
Loading train (GPU): /kaggle/input/datasets/anhtran10/lo-dataset/Median/Median/train_median.parquet
Train samples: 1859619
Validation samples: 232452
Phase 1: 39 features
Phase 2: 39 features
Phase 3: 39 features
Phase 4: 39 features
Time-series shape: (1859619, 4, 39)
Static feature shape: (1859619, 23)
Classes: [0 1 2]
Epoch 1: train = 0.2472, val = 0.1451
Epoch 2: train = 0.1644, val = 0.1219
Epoch 3: train = 0.1393, val = 0.1123
Epoch 4: train = 0.1241, val = 0.1031
Epoch 5: train = 0.1201, val = 0.1209
Epoch 6: train = 0.1074, val = 0.1108
Epoch 7: train = 0.1029, val = 0.1181
Epoch 8: train = 0.0947, val = 0.1354
Epoch 9: train = 0.0911, val = 0.1298
Epoch 10: train = 0.0891, val = 0.1225
Epoch 11: train = 0.0862, val = 0.1295
Epoch 12: train = 0.0827, val = 0.1147
Epoch 13: train = 0.0814, val = 0.1405
Epoch 14: train = 0.0812, val = 0.1417

--- Test Phase 1: /kaggle/input/datasets/anhtran10/lo-dataset/Median/Median

,Version,Phase,TimeBuildModel,TimePredict,Accuracy,BalancedAcc,Precision Macro,Precision Weighted,Recall Macro,Recall Weighted,F1-Score Macro,F1-Score Weighted,GMean,MCC,Kappa,Precision_Excellent,Recall_Excellent,F1-Score_Excellent,G-Mean_Excellent,Precision_Good,Recall_Good,F1-Score_Good,G-Mean_Good,Precision_Average,Recall_Average,F1-Score_Average,G-Mean_Average
0,V1 (Median),1,593.201622,3.595365,0.851359,0.713519,0.388454,0.996289,0.713519,0.851359,0.390679,0.916807,0.791225,0.117109,0.032581,0.150555,0.426009,0.222482,0.651940,0.015143,0.862810,0.029763,0.858176,0.999666,0.851739,0.919793,0.885351
1,V1 (Median),2,593.201622,3.514675,0.829032,0.794157,0.353157,0.996290,0.794157,0.829032,0.339918,0.903384,0.854281,0.114988,0.029705,0.045589,0.690583,0.085532,0.825225,0.014110,0.862810,0.027765,0.852686,0.999771,0.829077,0.906458,0.886014
2,V1 (Median),3,593.201622,3.518577,0.846227,0.880763,0.352106,0.996409,0.880763,0.846227,0.341818,0.913432,0.908847,0.131561,0.036145,0.038766,0.878924,0.074256,0.927648,0.017665,0.917355,0.034663,0.891762,0.999888,0.846010,0.916535,0.907486
3,V1 (Median),4,593.201622,3.512974,0.992558,0.946762,0.530385,0.997456,0.946762,0.992558,0.629691,0.994452,0.966555,0.547604,0.474016,0.250927,0.910314,0.393411,0.952858,0.340336,0.937190,0.499339,0.965789,0.999891,0.992781,0.996324,0.981227


In [ ]:
df_v1.to_csv("results_v1.csv", index=False)

## V_SMOTE

In [ ]:
base_path_1 = "/kaggle/input/datasets/uyentran10/lo-smote-test"

In [ ]:
base_path = "/kaggle/input/datasets/anhtran10/lo-dataset/Median/Median"

In [ ]:
df_v22 = run_experiment(
    base_path=base_path,
    train_file="train_median_smote.parquet",
    val_file="val.parquet",
    test_prefix="test",
    version_name="V22 (Median SMOTE)"
)
df_v22


####################
Version: V22 (Median SMOTE)
####################
Loading train (GPU): /kaggle/input/datasets/uyentran10/lo-smote-test/train_median_smote.parquet
Train samples: 5558991
Validation samples: 232452
Phase 1: 39 features
Phase 2: 39 features
Phase 3: 39 features
Phase 4: 39 features
Time-series shape: (5558991, 4, 39)
Static feature shape: (5558991, 23)
Classes: [0 1 2]
Epoch 1: train = 0.0473, val = 0.0311
Epoch 2: train = 0.0231, val = 0.0257
Epoch 3: train = 0.0190, val = 0.0284
Epoch 4: train = 0.0172, val = 0.0238
Epoch 5: train = 0.0159, val = 0.0255
Epoch 6: train = 0.0149, val = 0.0258
Epoch 7: train = 0.0141, val = 0.0267
Epoch 8: train = 0.0137, val = 0.0250
Epoch 9: train = 0.0132, val = 0.0284
Epoch 10: train = 0.0129, val = 0.0240
Epoch 11: train = 0.0125, val = 0.0252
Epoch 12: train = 0.0123, val = 0.0251
Epoch 13: train = 0.0120, val = 0.0275
Epoch 14: train = 0.0118, val = 0.0243

--- Test Phase 1: /kaggle/input/datasets/anhtran10/lo-dataset/Median/Med

,Version,Phase,TimeBuildModel,TimePredict,Accuracy,BalancedAcc,Precision Macro,Precision Weighted,Recall Macro,Recall Weighted,F1-Score Macro,F1-Score Weighted,GMean,MCC,Kappa,Precision_Excellent,Recall_Excellent,F1-Score_Excellent,G-Mean_Excellent,Precision_Good,Recall_Good,F1-Score_Good,G-Mean_Good,Precision_Average,Recall_Average,F1-Score_Average,G-Mean_Average
0,V22 (Median SMOTE),1,1718.235536,3.289717,0.993878,0.577514,0.501338,0.995019,0.577514,0.993878,0.457014,0.994058,0.489713,0.237527,0.235812,0.161957,0.668161,0.260717,0.816053,0.344538,0.067769,0.113260,0.260280,0.997520,0.996611,0.997065,0.552923
1,V22 (Median SMOTE),2,1718.235536,3.521524,0.996064,0.619220,0.616259,0.995785,0.619220,0.996064,0.582932,0.995724,0.601703,0.383933,0.381187,0.361179,0.659193,0.466667,0.811452,0.489879,0.200000,0.284038,0.447092,0.997718,0.998467,0.998092,0.600465
2,V22 (Median SMOTE),3,1718.235536,3.477125,0.996335,0.742466,0.650605,0.996722,0.742466,0.996335,0.664923,0.996415,0.756211,0.511115,0.510385,0.376033,0.816143,0.514851,0.902819,0.577367,0.413223,0.481696,0.642571,0.998415,0.998031,0.998223,0.745431
3,V22 (Median SMOTE),4,1718.235536,3.476438,0.994868,0.895795,0.584588,0.997351,0.895795,0.994868,0.679129,0.995823,0.921798,0.577469,0.538143,0.352734,0.896861,0.506329,0.946279,0.401503,0.795041,0.533555,0.890271,0.999528,0.995484,0.997502,0.929751


In [ ]:
df_v22.to_csv("results_v22.csv", index=False)

## V_GAN

In [ ]:
base_path_1 = "/kaggle/input/datasets/anhtran10/lo-gan-test"

In [ ]:
df_v23 = run_experiment(
    base_path=base_path,
    train_file="train_median_gan.parquet",
    val_file="val.parquet",
    test_prefix="test",
    version_name="V23 (Median GAN)"
)

df_v23


####################
Version: V23 (Median GAN)
####################
Loading train (GPU): /kaggle/input/datasets/anhtran10/lo-gan-test/train_median_gan.parquet
Train samples: 5558991
Validation samples: 232452
Phase 1: 39 features
Phase 2: 39 features
Phase 3: 39 features
Phase 4: 39 features
Time-series shape: (5558991, 4, 39)
Static feature shape: (5558991, 23)
Classes: [0 1 2]
Epoch 1: train = 0.0034, val = 0.0060
Epoch 2: train = 0.0021, val = 0.0052
Epoch 3: train = 0.0019, val = 0.0055
Epoch 4: train = 0.0018, val = 0.0050
Epoch 5: train = 0.0017, val = 0.0050
Epoch 6: train = 0.0017, val = 0.0050
Epoch 7: train = 0.0016, val = 0.0051
Epoch 8: train = 0.0015, val = 0.0049
Epoch 9: train = 0.0015, val = 0.0050
Epoch 10: train = 0.0015, val = 0.0050
Epoch 11: train = 0.0014, val = 0.0052
Epoch 12: train = 0.0014, val = 0.0050
Epoch 13: train = 0.0014, val = 0.0052
Epoch 14: train = 0.0013, val = 0.0051

--- Test Phase 1: /kaggle/input/datasets/anhtran10/lo-dataset/Median/Median/tes

,Version,Phase,TimeBuildModel,TimePredict,Accuracy,BalancedAcc,Precision Macro,Precision Weighted,Recall Macro,Recall Weighted,F1-Score Macro,F1-Score Weighted,GMean,MCC,Kappa,Precision_Excellent,Recall_Excellent,F1-Score_Excellent,G-Mean_Excellent,Precision_Good,Recall_Good,F1-Score_Good,G-Mean_Good,Precision_Average,Recall_Average,F1-Score_Average,G-Mean_Average
0,V23 (Median GAN),1,1730.613686,3.548552,0.996051,0.423351,0.748521,0.995392,0.423351,0.996051,0.455503,0.995327,0.377954,0.275684,0.259956,0.933333,0.062780,0.117647,0.250559,0.315000,0.208264,0.250746,0.456090,0.997229,0.999007,0.998117,0.472449
1,V23 (Median GAN),2,1730.613686,3.505948,0.996782,0.454775,0.811550,0.996146,0.454775,0.996782,0.506652,0.996049,0.447916,0.401114,0.368153,0.956522,0.098655,0.178862,0.314093,0.480597,0.266116,0.342553,0.515670,0.997531,0.999555,0.998542,0.554828
2,V23 (Median GAN),3,1730.613686,3.495813,0.996163,0.587376,0.781258,0.996853,0.587376,0.996163,0.606438,0.996262,0.648786,0.488183,0.487484,0.982143,0.246637,0.394265,0.496624,0.363109,0.517355,0.426721,0.718422,0.998523,0.998135,0.998329,0.765416
3,V23 (Median GAN),4,1730.613686,3.493025,0.998486,0.823034,0.889039,0.998368,0.823034,0.998486,0.852782,0.998402,0.852647,0.771141,0.768127,0.878505,0.843049,0.860412,0.918126,0.789583,0.626446,0.698618,0.791311,0.999029,0.999607,0.999318,0.853214


In [ ]:
df_v23.to_csv("results_v23.csv", index=False)

## V_CDSMOTE

In [ ]:
df_v2 = run_experiment(
    base_path=base_path,
    train_file="train_median_cdsmote.parquet",
    val_file="val.parquet",
    test_prefix="test",
    version_name="V2 (Median CDSMOTE)"
)

df_v2


####################
Version: V2 (Median CDSMOTE)
####################
Loading train (GPU): /kaggle/input/datasets/anhtran10/lo-dataset/Median/Median/train_median_cdsmote.parquet
Train samples: 5558987
Validation samples: 232452
Phase 1: 39 features
Phase 2: 39 features
Phase 3: 39 features
Phase 4: 39 features
Time-series shape: (5558987, 4, 39)
Static feature shape: (5558987, 23)
Classes: [0 1 2]
Epoch 1: train = 0.0347, val = 0.0233
Epoch 2: train = 0.0173, val = 0.0228
Epoch 3: train = 0.0145, val = 0.0207
Epoch 4: train = 0.0129, val = 0.0223
Epoch 5: train = 0.0119, val = 0.0201
Epoch 6: train = 0.0112, val = 0.0198
Epoch 7: train = 0.0107, val = 0.0188
Epoch 8: train = 0.0102, val = 0.0212
Epoch 9: train = 0.0098, val = 0.0189
Epoch 10: train = 0.0096, val = 0.0192
Epoch 11: train = 0.0093, val = 0.0214
Epoch 12: train = 0.0090, val = 0.0178
Epoch 13: train = 0.0089, val = 0.0217
Epoch 14: train = 0.0087, val = 0.0190
Epoch 15: train = 0.0086, val = 0.0199
Epoch 16: train = 0.0

,Version,Phase,TimeBuildModel,TimePredict,Accuracy,BalancedAcc,Precision Macro,Precision Weighted,Recall Macro,Recall Weighted,F1-Score Macro,F1-Score Weighted,GMean,MCC,Kappa,Precision_Excellent,Recall_Excellent,F1-Score_Excellent,G-Mean_Excellent,Precision_Good,Recall_Good,F1-Score_Good,G-Mean_Good,Precision_Average,Recall_Average,F1-Score_Average,G-Mean_Average
0,V2 (Median CDSMOTE),1,2734.581369,3.600424,0.996567,0.376838,0.987363,0.996550,0.376838,0.996567,0.410124,0.994963,0.169015,0.190271,0.071932,0.965517,0.125561,0.222222,0.354344,1.000000,0.004959,0.009868,0.070418,0.996571,0.999996,0.998280,0.193493
1,V2 (Median CDSMOTE),2,2734.581369,3.670368,0.996799,0.438441,0.955886,0.996602,0.438441,0.996799,0.501569,0.995475,0.332588,0.318671,0.192643,0.937500,0.269058,0.418118,0.518704,0.933333,0.046281,0.088189,0.215129,0.996824,0.999983,0.998401,0.329687
2,V2 (Median CDSMOTE),3,2734.581369,3.539086,0.997230,0.698552,0.780550,0.997219,0.698552,0.997230,0.691899,0.996898,0.699330,0.548648,0.539351,0.507331,0.775785,0.613475,0.880468,0.836207,0.320661,0.463560,0.566223,0.998111,0.999210,0.998660,0.686034
3,V2 (Median CDSMOTE),4,2734.581369,3.542655,0.995466,0.896856,0.615491,0.997502,0.896856,0.995466,0.709030,0.996240,0.924986,0.605758,0.571653,0.436123,0.887892,0.584934,0.941761,0.410774,0.806612,0.544339,0.896759,0.999575,0.996063,0.997816,0.937108


In [ ]:
df_v2.to_csv("results_v2.csv", index=False)

## V_SMOTified GAN

In [ ]:
base_path_1 = "/kaggle/input/datasets/uyentran10/lo-smotifiedgan-test"

In [ ]:
df_v24 = run_experiment(
    base_path=base_path,
    train_file="train_median_smotified_gan.parquet",
    val_file="val.parquet",
    test_prefix="test",
    version_name="V24 (Median SMOTified GAN)"
)

df_v24


####################
Version: V24 (Median SMOTified GAN)
####################
Loading train (GPU): /kaggle/input/datasets/uyentran10/lo-smotifiedgan-test/train_median_smotified_gan.parquet
Train samples: 5558991
Validation samples: 232452
Phase 1: 39 features
Phase 2: 39 features
Phase 3: 39 features
Phase 4: 39 features
Time-series shape: (5558991, 4, 39)
Static feature shape: (5558991, 23)
Classes: [0 1 2]
Epoch 1: train = 0.0036, val = 0.0063
Epoch 2: train = 0.0021, val = 0.0052
Epoch 3: train = 0.0019, val = 0.0048
Epoch 4: train = 0.0018, val = 0.0050
Epoch 5: train = 0.0017, val = 0.0050
Epoch 6: train = 0.0016, val = 0.0051
Epoch 7: train = 0.0016, val = 0.0049
Epoch 8: train = 0.0015, val = 0.0051
Epoch 9: train = 0.0015, val = 0.0048
Epoch 10: train = 0.0014, val = 0.0050
Epoch 11: train = 0.0014, val = 0.0049
Epoch 12: train = 0.0014, val = 0.0048
Epoch 13: train = 0.0014, val = 0.0049

--- Test Phase 1: /kaggle/input/datasets/anhtran10/lo-dataset/Median/Median/test_1.parqu

,Version,Phase,TimeBuildModel,TimePredict,Accuracy,BalancedAcc,Precision Macro,Precision Weighted,Recall Macro,Recall Weighted,F1-Score Macro,F1-Score Weighted,GMean,MCC,Kappa,Precision_Excellent,Recall_Excellent,F1-Score_Excellent,G-Mean_Excellent,Precision_Good,Recall_Good,F1-Score_Good,G-Mean_Good,Precision_Average,Recall_Average,F1-Score_Average,G-Mean_Average
0,V24 (Median SMOTified GAN),1,1626.688283,3.350028,0.946295,0.613029,0.629480,0.996739,0.613029,0.946295,0.409271,0.969658,0.627506,0.175907,0.081913,0.851852,0.103139,0.184000,0.321150,0.037202,0.788430,0.071051,0.863973,0.999385,0.947519,0.972761,0.890523
1,V24 (Median SMOTified GAN),2,1626.688283,3.514425,0.892546,0.652857,0.452304,0.996459,0.652857,0.892546,0.406220,0.940427,0.697736,0.135490,0.044859,0.336134,0.179372,0.233918,0.423452,0.021135,0.885950,0.041285,0.889431,0.999642,0.893250,0.943456,0.901897
2,V24 (Median SMOTified GAN),3,1626.688283,3.524996,0.955819,0.765285,0.561792,0.996798,0.765285,0.955819,0.539120,0.974884,0.834981,0.223495,0.112714,0.636905,0.479821,0.547315,0.692600,0.048849,0.859504,0.092444,0.906625,0.999621,0.956529,0.977600,0.927087
3,V24 (Median SMOTified GAN),4,1626.688283,3.520493,0.998451,0.829378,0.878708,0.998357,0.829378,0.998451,0.852338,0.998391,0.859937,0.769622,0.768021,0.865116,0.834081,0.849315,0.913223,0.771930,0.654545,0.708408,0.808836,0.999076,0.999508,0.999292,0.860919


In [ ]:
df_v24.to_csv("results_v24.csv", index=False)

## V_CDSMOTified GAN

In [ ]:
base_path_1 = "/kaggle/input/datasets/hngliththu/cdsmote-gan"

In [ ]:
base_path = "/kaggle/input/datasets/anhtran10/lo-dataset/Median/Median"

In [ ]:
df_v13 = run_experiment(
    base_path=base_path,
    train_file="train_median_cdsmote_gan.parquet",
    val_file="val.parquet",
    test_prefix="test",
    version_name="V13 (Median CDSMOTified GAN)"
)
df_v13


####################
Version: V13 (Median CDSMOTified GAN)
####################
Loading train (GPU): /kaggle/input/datasets/hngliththu/cdsmote-gan/train_median_cdsmote_gan.parquet
Train samples: 5558987
Validation samples: 232452
Phase 1: 39 features
Phase 2: 39 features
Phase 3: 39 features
Phase 4: 39 features
Time-series shape: (5558987, 4, 39)
Static feature shape: (5558987, 23)
Classes: [0 1 2]
Epoch 1: train = 0.0035, val = 0.0057
Epoch 2: train = 0.0020, val = 0.0050
Epoch 3: train = 0.0019, val = 0.0050
Epoch 4: train = 0.0017, val = 0.0047
Epoch 5: train = 0.0017, val = 0.0054
Epoch 6: train = 0.0016, val = 0.0045
Epoch 7: train = 0.0015, val = 0.0046
Epoch 8: train = 0.0015, val = 0.0047
Epoch 9: train = 0.0015, val = 0.0053
Epoch 10: train = 0.0014, val = 0.0052
Epoch 11: train = 0.0014, val = 0.0049
Epoch 12: train = 0.0013, val = 0.0051
Epoch 13: train = 0.0013, val = 0.0050
Epoch 14: train = 0.0013, val = 0.0057
Epoch 15: train = 0.0013, val = 0.0055
Epoch 16: train = 0.

,Version,Phase,TimeBuildModel,TimePredict,Accuracy,BalancedAcc,Precision Macro,Precision Weighted,Recall Macro,Recall Weighted,F1-Score Macro,F1-Score Weighted,GMean,MCC,Kappa,Precision_Excellent,Recall_Excellent,F1-Score_Excellent,G-Mean_Excellent,Precision_Good,Recall_Good,F1-Score_Good,G-Mean_Good,Precision_Average,Recall_Average,F1-Score_Average,G-Mean_Average
0,V13 (Median CDSMOTified GAN),1,1930.454707,3.456331,0.994881,0.504553,0.730642,0.995772,0.504553,0.994881,0.513268,0.995006,0.516847,0.322276,0.321670,0.937500,0.134529,0.235294,0.366781,0.256667,0.381818,0.306977,0.617022,0.997758,0.997310,0.997534,0.610069
1,V13 (Median CDSMOTified GAN),2,1930.454707,3.338682,0.996292,0.574708,0.785849,0.996460,0.574708,0.996292,0.615049,0.996185,0.616542,0.451420,0.450782,0.951613,0.264574,0.414035,0.514364,0.407895,0.461157,0.432894,0.678492,0.998041,0.998394,0.998217,0.671540
2,V13 (Median CDSMOTified GAN),3,1930.454707,3.449917,0.996963,0.731264,0.787601,0.997437,0.731264,0.996963,0.740691,0.997122,0.791095,0.607774,0.605862,0.889706,0.542601,0.674095,0.736591,0.474190,0.652893,0.549374,0.807254,0.998907,0.998299,0.998603,0.832624
3,V13 (Median CDSMOTified GAN),4,1930.454707,3.402359,0.998404,0.839589,0.863716,0.998310,0.839589,0.998404,0.849074,0.998339,0.863968,0.763735,0.762460,0.825000,0.887892,0.855292,0.942195,0.767068,0.631405,0.692656,0.794411,0.999081,0.999469,0.999275,0.861603


In [ ]:
df_v13.to_csv("results_v13.csv", index=False)